# Lab 6 Task 2 - Connect Four con Minimax

## Que vas a ver aqui
1. Una clase `Connect4` para manejar el tablero.
2. Funciones para detectar victorias y estados terminales.
3. Un agente Minimax puro (sin poda) para elegir jugadas.

## Regla de puntuacion usada
- Gana IA: valor alto positivo.
- Gana jugador: valor alto negativo.
- Empate: 0.

## Nota importante
La profundidad se limita a `d = 3` o `d = 4` porque Minimax puro crece muy rapido en tiempo de calculo.

In [ ]:
# imports y  constantes 
import math
import copy
import random


ROWS = 6
COLS = 7
EMPTY = 0
PLAYER = 1
AI = 2

## Task 2.1 Class Connect4

Esta clase manejará:
- el estado del tablero
- los movimientos válidos
- la colocación de fichas
- la detección de victoria
- la verificación de empate o estado terminal

Se usará una matriz de 6 filas por 7 columnas, donde:
- `0` representa una casilla vacía
- `1` representa una ficha del jugador
- `2` representa una ficha de la IA

In [ ]:
class Connect4:
    def __init__(self):
        self.board = [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]

    def clone(self):
        new_game = Connect4()
        new_game.board = copy.deepcopy(self.board)
        return new_game

    def print_board(self):
        for row in self.board:
            print(row)
        print("0 1 2 3 4 5 6")
        print()

    def actions(self):
        """
        Devuelve una lista con las columnas válidas donde aún se puede jugar.
        """
        valid_moves = []
        for col in range(COLS):
            if self.board[0][col] == EMPTY:
                valid_moves.append(col)
        return valid_moves

    def drop_piece(self, col, piece):
        """
        Coloca una ficha en la columna indicada.
        La ficha cae hasta la posición más baja disponible.
        Devuelve True si se pudo colocar, False si la jugada no es válida.
        """
        if col < 0 or col >= COLS or self.board[0][col] != EMPTY:
            return False

        for row in range(ROWS - 1, -1, -1):
            if self.board[row][col] == EMPTY:
                self.board[row][col] = piece
                return True

        return False

    def is_full(self):
        """
        Devuelve True si ya no hay movimientos válidos.
        """
        return len(self.actions()) == 0

    def check_winner(self, piece):
        """
        Revisa si la ficha indicada tiene 4 en línea.
        """

        # Horizontal
        for row in range(ROWS):
            for col in range(COLS - 3):
                if (
                    self.board[row][col] == piece and
                    self.board[row][col + 1] == piece and
                    self.board[row][col + 2] == piece and
                    self.board[row][col + 3] == piece
                ):
                    return True

        # Vertical
        for row in range(ROWS - 3):
            for col in range(COLS):
                if (
                    self.board[row][col] == piece and
                    self.board[row + 1][col] == piece and
                    self.board[row + 2][col] == piece and
                    self.board[row + 3][col] == piece
                ):
                    return True

        # Diagonal descendente hacia la derecha
        for row in range(ROWS - 3):
            for col in range(COLS - 3):
                if (
                    self.board[row][col] == piece and
                    self.board[row + 1][col + 1] == piece and
                    self.board[row + 2][col + 2] == piece and
                    self.board[row + 3][col + 3] == piece
                ):
                    return True

        # Diagonal ascendente hacia la derecha
        for row in range(3, ROWS):
            for col in range(COLS - 3):
                if (
                    self.board[row][col] == piece and
                    self.board[row - 1][col + 1] == piece and
                    self.board[row - 2][col + 2] == piece and
                    self.board[row - 3][col + 3] == piece
                ):
                    return True

        return False

    def is_terminal(self):
        """
        Un estado terminal ocurre si:
        - gana PLAYER
        - gana AI
        - o el tablero está lleno
        """
        return self.check_winner(PLAYER) or self.check_winner(AI) or self.is_full()

## Prueba básica del tablero

En esta parte se prueba:
- crear un tablero vacío
- colocar fichas
- mostrar el tablero
- verificar movimientos válidos

In [ ]:
game = Connect4()
game.print_board()

game.drop_piece(3, PLAYER)
game.drop_piece(3, AI)
game.drop_piece(2, PLAYER)
game.drop_piece(4, AI)

game.print_board()
print("Movimientos válidos:", game.actions())
print("¿Es terminal?", game.is_terminal())

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
0 1 2 3 4 5 6

[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 1, 1, 2, 0, 0]
0 1 2 3 4 5 6

Movimientos válidos: [0, 1, 2, 3, 4, 5, 6]
¿Es terminal? False


## Task 2.2: Agente Alfa-Beta (Profundidad 5 o 6)

En este bloque implementamos Alfa-Beta con una heurística sencilla y explicable.

### Estrategia heurística (resumen para tu video)
- Favorecer jugar en el centro del tablero.
- Dar puntaje alto a ventanas de 4 casillas con:
  - 3 fichas de IA + 1 vacía (amenaza fuerte).
  - 2 fichas de IA + 2 vacías (potencial).
- Penalizar ventanas donde el rival tiene 3 fichas + 1 vacía (bloqueo defensivo).
- Estados terminales:
  - Victoria IA: valor muy alto positivo.
  - Victoria rival: valor muy alto negativo.
  - Empate: 0.

Con esto, Alfa-Beta profundidad 5 o 6 toma decisiones fuertes sin explorar todos los nodos de Minimax puro.

In [ ]:
alpha_beta_nodes_visited = 0


def evaluate_window(window, piece):
    """Evalua una ventana de 4 casillas para construir la heuristica."""
    score = 0
    opp_piece = PLAYER if piece == AI else AI

    if window.count(piece) == 4:
        score += 100
    elif window.count(piece) == 3 and window.count(EMPTY) == 1:
        score += 10
    elif window.count(piece) == 2 and window.count(EMPTY) == 2:
        score += 4

    # Defensa: penaliza dejar amenazas claras del rival.
    if window.count(opp_piece) == 3 and window.count(EMPTY) == 1:
        score -= 12

    return score


def score_position(board, piece):
    """Heuristica total del tablero para la IA."""
    score = 0
    grid = board.board

    # Prioriza la columna central.
    center_col = COLS // 2
    center_array = [grid[r][center_col] for r in range(ROWS)]
    score += center_array.count(piece) * 3

    # Horizontal
    for r in range(ROWS):
        row_array = grid[r]
        for c in range(COLS - 3):
            window = row_array[c:c + 4]
            score += evaluate_window(window, piece)

    # Vertical
    for c in range(COLS):
        col_array = [grid[r][c] for r in range(ROWS)]
        for r in range(ROWS - 3):
            window = col_array[r:r + 4]
            score += evaluate_window(window, piece)

    # Diagonal descendente
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            window = [grid[r + i][c + i] for i in range(4)]
            score += evaluate_window(window, piece)

    # Diagonal ascendente
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            window = [grid[r - i][c + i] for i in range(4)]
            score += evaluate_window(window, piece)

    return score


def ordered_actions(board):
    """Ordena jugadas desde el centro para podar mejor en Alfa-Beta."""
    center = COLS // 2
    return sorted(board.actions(), key=lambda c: abs(c - center))


def alpha_beta(board, depth, alpha, beta, maximizing_player):
    """Busqueda Minimax con poda Alfa-Beta."""
    global alpha_beta_nodes_visited
    alpha_beta_nodes_visited += 1

    valid_locations = board.actions()
    is_term = board.is_terminal()

    if depth == 0 or is_term:
        if is_term:
            if board.check_winner(AI):
                return 1000000000
            if board.check_winner(PLAYER):
                return -1000000000
            return 0
        return score_position(board, AI)

    # Ordenar movimientos mejora el efecto de la poda.
    valid_locations = ordered_actions(board)

    if maximizing_player:
        value = -math.inf
        for col in valid_locations:
            b_copy = board.clone()
            b_copy.drop_piece(col, AI)
            value = max(value, alpha_beta(b_copy, depth - 1, alpha, beta, False))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value

    value = math.inf
    for col in valid_locations:
        b_copy = board.clone()
        b_copy.drop_piece(col, PLAYER)
        value = min(value, alpha_beta(b_copy, depth - 1, alpha, beta, True))
        beta = min(beta, value)
        if alpha >= beta:
            break
    return value


def get_best_move_alpha_beta(board, depth=5):
    """Mejor jugada de la IA usando Alfa-Beta."""
    global alpha_beta_nodes_visited
    alpha_beta_nodes_visited = 0

    valid_locations = ordered_actions(board)
    if not valid_locations:
        return None

    best_score = -math.inf
    best_col = valid_locations[0]
    alpha = -math.inf
    beta = math.inf

    for col in valid_locations:
        b_copy = board.clone()
        b_copy.drop_piece(col, AI)
        score = alpha_beta(b_copy, depth - 1, alpha, beta, False)

        if score > best_score:
            best_score = score
            best_col = col

        alpha = max(alpha, best_score)

    return best_col


def play_alpha_beta_vs_random(depth=5, verbose=True):
    """Partida simple: IA (Alfa-Beta) vs agente aleatorio."""
    game = Connect4()
    turn = AI  # IA comienza para la demo del video.

    if verbose:
        print("Inicio: IA (Alfa-Beta) vs Aleatorio")
        game.print_board()

    while not game.is_terminal():
        if turn == AI:
            col = get_best_move_alpha_beta(game, depth)
            label = "IA"
        else:
            col = random.choice(game.actions())
            label = "Aleatorio"

        game.drop_piece(col, turn)

        if verbose:
            print(f"{label} juega columna {col}")
            game.print_board()

        turn = PLAYER if turn == AI else AI

    if game.check_winner(AI):
        print("Resultado: Gana IA (Alfa-Beta)")
        return AI
    if game.check_winner(PLAYER):
        print("Resultado: Gana Aleatorio")
        return PLAYER

    print("Resultado: Empate")
    return EMPTY


def benchmark_alpha_beta_vs_random(num_games=20, depth=5):
    """Ejecuta varias partidas para mostrar que la IA gana consistentemente."""
    wins = 0
    losses = 0
    draws = 0

    for _ in range(num_games):
        game = Connect4()
        turn = AI

        while not game.is_terminal():
            if turn == AI:
                col = get_best_move_alpha_beta(game, depth)
            else:
                col = random.choice(game.actions())

            game.drop_piece(col, turn)
            turn = PLAYER if turn == AI else AI

        if game.check_winner(AI):
            wins += 1
        elif game.check_winner(PLAYER):
            losses += 1
        else:
            draws += 1

    print(f"Benchmark IA vs Aleatorio | partidas={num_games}, profundidad={depth}")
    print(f"IA gana: {wins} | IA pierde: {losses} | Empates: {draws}")
    print(f"Win rate IA: {wins / num_games:.2%}")


def play_human_vs_alpha_beta(depth=5, human_piece=PLAYER):
    """Modo interactivo: tu juegas contra la IA en consola/notebook."""
    game = Connect4()
    ai_piece = AI if human_piece == PLAYER else PLAYER
    turn = PLAYER

    print("Tu vs IA (Alfa-Beta)")
    print(f"Tu ficha: {human_piece} | IA ficha: {ai_piece}")
    game.print_board()

    while not game.is_terminal():
        if turn == human_piece:
            valid = game.actions()
            col_text = input(f"Tu turno. Elige columna {valid}: ")
            try:
                col = int(col_text)
            except ValueError:
                print("Entrada invalida. Escribe un numero de columna.")
                continue

            if col not in valid:
                print("Columna no valida. Intenta otra vez.")
                continue

            game.drop_piece(col, human_piece)
            print(f"Tu juegas columna {col}")
        else:
            col = get_best_move_alpha_beta(game, depth)
            game.drop_piece(col, ai_piece)
            print(f"IA juega columna {col}")

        game.print_board()
        turn = ai_piece if turn == human_piece else human_piece

    if game.check_winner(human_piece):
        print("Resultado final: Ganaste")
    elif game.check_winner(ai_piece):
        print("Resultado final: Gana la IA")
    else:
        print("Resultado final: Empate")


print("Mejor columna Alfa-Beta (d=5):", get_best_move_alpha_beta(game, 5))
print("Nodos visitados Alfa-Beta:", alpha_beta_nodes_visited)

Mejor columna Alfa-Beta (d=5): 3
Nodos visitados Alfa-Beta: 1721


## Task 2.2: Comparacion Minimax Puro vs Alfa-Beta

En esta celda usamos el mismo estado del tablero y la misma profundidad para comparar cuantos nodos explora cada algoritmo.

Esperado: Alfa-Beta visita muchos menos nodos.

In [ ]:
minimax_nodes_visited = 0


def minimax_pure(board, depth, maximizing_player):
    """Minimax recursivo sin poda para Task 2.1."""
    global minimax_nodes_visited
    minimax_nodes_visited += 1

    valid_locations = board.actions()
    is_term = board.is_terminal()

    if depth == 0 or is_term:
        if is_term:
            if board.check_winner(AI):
                return 1000000000
            if board.check_winner(PLAYER):
                return -1000000000
            return 0
        return score_position(board, AI)

    if maximizing_player:
        value = -math.inf
        for col in valid_locations:
            b_copy = board.clone()
            b_copy.drop_piece(col, AI)
            value = max(value, minimax_pure(b_copy, depth - 1, False))
        return value

    value = math.inf
    for col in valid_locations:
        b_copy = board.clone()
        b_copy.drop_piece(col, PLAYER)
        value = min(value, minimax_pure(b_copy, depth - 1, True))
    return value


def get_best_move_minimax(board, depth=4):
    """Mejor jugada usando Minimax puro (usar d=3 o d=4)."""
    global minimax_nodes_visited
    minimax_nodes_visited = 0

    valid_locations = board.actions()
    if not valid_locations:
        return None

    best_score = -math.inf
    best_col = valid_locations[0]

    for col in valid_locations:
        b_copy = board.clone()
        b_copy.drop_piece(col, AI)
        score = minimax_pure(b_copy, depth - 1, False)
        if score > best_score:
            best_score = score
            best_col = col

    return best_col


def get_best_move(board, depth):
    return get_best_move_minimax(board, depth)


def compare_minimax_vs_alpha_beta(board, depth=4):
    """Task 2.2: compara nodos en el mismo estado y profundidad."""
    best_minimax = get_best_move_minimax(board, depth)
    nodes_minimax = minimax_nodes_visited

    best_ab = get_best_move_alpha_beta(board, depth)
    nodes_ab = alpha_beta_nodes_visited

    print(f"Comparacion en profundidad {depth}")
    print(f"Minimax puro -> mejor columna: {best_minimax}, nodos: {nodes_minimax}")
    print(f"Alfa-Beta    -> mejor columna: {best_ab}, nodos: {nodes_ab}")

    if nodes_minimax > 0:
        reduction = (1 - (nodes_ab / nodes_minimax)) * 100
        print(f"Reduccion de nodos con Alfa-Beta: {reduction:.2f}%")


compare_minimax_vs_alpha_beta(game, depth=4)

Comparacion en profundidad 4
Minimax puro -> mejor columna: 1, nodos: 2800
Alfa-Beta    -> mejor columna: 4, nodos: 625
Reduccion de nodos con Alfa-Beta: 77.68%


## PRESENTACION

1. Comparacion Minimax vs Alfa-Beta (Task 2.2).
2. Benchmark rapido IA vs Aleatorio (profundidad 5 o 6).
3. Partida visible IA vs Aleatorio para mostrar jugadas.
4. Tu contra la IA.



In [ ]:
# 1) Task 2.2: Comparacion de nodos (Minimax vs Alfa-Beta)
compare_minimax_vs_alpha_beta(game, depth=4)

# 2) Evidencia rapida: IA vs Aleatorio en varias partidas
benchmark_alpha_beta_vs_random(num_games=20, depth=5)

# 3) Una partida visible (paso a paso)
play_alpha_beta_vs_random(depth=5, verbose=True)


Comparacion en profundidad 4
Minimax puro -> mejor columna: 1, nodos: 2800
Alfa-Beta    -> mejor columna: 4, nodos: 625
Reduccion de nodos con Alfa-Beta: 77.68%
Benchmark IA vs Aleatorio | partidas=20, profundidad=5
IA gana: 20 | IA pierde: 0 | Empates: 0
Win rate IA: 100.00%
Inicio: IA (Alfa-Beta) vs Aleatorio
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
0 1 2 3 4 5 6

IA juega columna 3
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
0 1 2 3 4 5 6

Aleatorio juega columna 4
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 1, 0, 0]
0 1 2 3 4 5 6

IA juega columna 3
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 0, 0]
[0, 0, 0, 2, 1, 0, 0]
0 1 2 3 4 5 6

Aleatorio juega columna 4
[0, 0, 0,

2

In [ ]:

# 4) Partida vs IA 
# play_human_vs_alpha_beta(depth=5, human_piece=PLAYER)

Tu vs IA (Alfa-Beta)
Tu ficha: 1 | IA ficha: 2
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
0 1 2 3 4 5 6

Tu juegas columna 5
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 1, 0]
0 1 2 3 4 5 6

IA juega columna 3
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 1, 0]
0 1 2 3 4 5 6

Tu juegas columna 5
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 1, 0]
[0, 0, 0, 2, 0, 1, 0]
0 1 2 3 4 5 6

IA juega columna 3
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 1, 0]
[0, 0, 0, 2, 0, 1, 0]
0 1 2 3 4 5 6

Tu juegas columna 1
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 2, 0, 1, 0]
[0, 1, 0, 

## Lab 7 - Task 2.1: TD Learning para Connect Four

En este laboratorio se agrega un agente TD que aprende por experiencia.

### Representacion del estado
- Tabular: el tablero se aplana a una tupla de 42 valores (0, 1, 2).
- Justificacion: simple, costo por paso bajo, y se guarda solo lo visitado (diccionario esparso).

### Algoritmo de actualizacion (Q-learning)
- Se aprende $Q(s,a)$ con actualizacion off-policy:
$$
Q(s,a) \leftarrow Q(s,a) + \alpha (r + \gamma \max_{a'} Q(s',a') - Q(s,a))
$$
- Esta implementacion trabaja con acciones y valores Q, no con $V^\pi(s)$.

### Funcion de recompensa
- Gana IA: +1
- Pierde IA: -1
- Empate: 0
- Recompensas intermedias: 0 en estados no terminales.

### Exploracion
- Epsilon-greedy con decaimiento desde 1.0 hasta 0.05.

### Ciclo de entrenamiento y evaluacion
- Entrenamiento contra oponente fijo (aleatorio) por N episodios.
- Se reporta win rate por bloques y luego se evalua con epsilon=0.

In [ ]:
from collections import defaultdict


def board_to_key(board):
    return tuple(cell for row in board.board for cell in row)


class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.95):
        self.alpha = alpha
        self.gamma = gamma
        self.q = defaultdict(lambda: [0.0 for _ in range(COLS)])

    def select_action(self, board, epsilon=0.0):
        valid = board.actions()
        if not valid:
            return None

        if random.random() < epsilon:
            return random.choice(valid)

        state = board_to_key(board)
        q_values = self.q[state]
        max_q = max(q_values[a] for a in valid)
        best = [a for a in valid if q_values[a] == max_q]
        return random.choice(best)

    def update(self, state, action, reward, next_state, done, next_actions):
        current_q = self.q[state][action]
        if done or not next_actions:
            target = reward
        else:
            next_max = max(self.q[next_state][a] for a in next_actions)
            target = reward + self.gamma * next_max

        self.q[state][action] = current_q + self.alpha * (target - current_q)


def train_q_agent(
    episodes=3000,
    alpha=0.1,
    gamma=0.95,
    epsilon=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.995,
    report_every=300,
    seed=7,
):
    random.seed(seed)
    agent = QLearningAgent(alpha=alpha, gamma=gamma)
    history = []
    wins = 0
    losses = 0
    draws = 0

    for ep in range(1, episodes + 1):
        game = Connect4()

        while not game.is_terminal():
            state = board_to_key(game)
            action = agent.select_action(game, epsilon)
            game.drop_piece(action, AI)

            if game.is_terminal():
                if game.check_winner(AI):
                    reward = 1
                    wins += 1
                else:
                    reward = 0
                    draws += 1

                agent.update(state, action, reward, board_to_key(game), True, [])
                break

            opp_action = random.choice(game.actions())
            game.drop_piece(opp_action, PLAYER)

            if game.is_terminal():
                if game.check_winner(PLAYER):
                    reward = -1
                    losses += 1
                else:
                    reward = 0
                    draws += 1
                done = True
            else:
                reward = 0
                done = False

            next_state = board_to_key(game)
            next_actions = game.actions()
            agent.update(state, action, reward, next_state, done, next_actions)

            if done:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        if ep % report_every == 0:
            total = wins + losses + draws
            win_rate = (wins / total) if total else 0.0
            history.append((ep, wins, losses, draws, win_rate))
            print(
                f"Ep {ep} | win {wins} loss {losses} draw {draws} | win_rate {win_rate:.2%}"
            )
            wins = 0
            losses = 0
            draws = 0

    return agent, history


def evaluate_agent(agent, num_games=200, seed=99):
    random.seed(seed)
    wins = 0
    losses = 0
    draws = 0

    for _ in range(num_games):
        game = Connect4()

        while not game.is_terminal():
            action = agent.select_action(game, epsilon=0.0)
            game.drop_piece(action, AI)

            if game.is_terminal():
                if game.check_winner(AI):
                    wins += 1
                else:
                    draws += 1
                break

            opp_action = random.choice(game.actions())
            game.drop_piece(opp_action, PLAYER)

            if game.is_terminal():
                if game.check_winner(PLAYER):
                    losses += 1
                else:
                    draws += 1
                break

    win_rate = wins / num_games if num_games else 0.0
    print(
        f"Eval vs random | games={num_games} wins={wins} losses={losses} "
        f"draws={draws} win_rate={win_rate:.2%}"
    )

## Evaluacion del agente
- Durante el entrenamiento se imprime el win rate por bloques (report_every).
- La evaluacion final usa epsilon=0 para medir rendimiento puro.
- Si el win rate aumenta con los episodios, el agente esta aprendiendo.

In [ ]:
td_agent, td_history = train_q_agent(episodes=3000, report_every=300)
evaluate_agent(td_agent, num_games=200)